- Note: merge()/join() columns ko side-by-side jodte hain (common key ke through), lekin concat() DataFrames ko upar-neeche (rows) ya bagal-bagal (columns) stack karta hai — bina kisi key ki zaroorat ke. Yeh SQL ke UNION jaisa concept hai (Day 8 mein FULL OUTER JOIN workaround mein use kiya tha).

1. Do DataFrames ko rows mein stack karna (vertical concat, sabse common use):

In [1]:
import pandas as pd

batch1 = pd.DataFrame({
    'Customer ID': [12346, 12347],
    'Country': ['United Kingdom', 'Germany']
})

batch2 = pd.DataFrame({
    'Customer ID': [12348, 12349],
    'Country': ['France', 'Spain']
})

combined = pd.concat([batch1, batch2])
print(combined)
print(combined.shape)

   Customer ID         Country
0        12346  United Kingdom
1        12347         Germany
0        12348          France
1        12349           Spain
(4, 2)


2. Index reset karna (concat ke baad index duplicate ho sakta hai — dekho upar wale output mein 0,1,0,1):

In [2]:
combined_clean = pd.concat([batch1, batch2], ignore_index=True)
print(combined_clean)

   Customer ID         Country
0        12346  United Kingdom
1        12347         Germany
2        12348          France
3        12349           Spain


Keep mind: ignore_index=True bahut zaroori hai jab hum multiple batches ko jodte ho, warna baad mein .loc[0] jaise operations confusing ho jaate hain (multiple rows same index 0 ki hongi).

3. Real use-case — agar humhara data multiple files/years mein bata hota (jaise 2 alag Excel sheets, jaisa original PDF ne mention kiya tha 2009-2010 aur 2010-2011):

In [3]:
# Simulate karte hain ki tumhare paas 2 alag "yearly" files hote
df_year1 = pd.DataFrame({'Invoice': ['489434', '489435'], 'Year': [2009, 2009]})
df_year2 = pd.DataFrame({'Invoice': ['537226', '537227'], 'Year': [2010, 2010]})

full_data = pd.concat([df_year1, df_year2], ignore_index=True)
print(full_data)

  Invoice  Year
0  489434  2009
1  489435  2009
2  537226  2010
3  537227  2010


Note: Tumhare paas already combined CSV hai (Day 3 mein dekha tha), isliye yeh sirf concept-practice hai — agar kabhi alag files milein to yehi pattern use hoga.

4. Columns mein concat karna (horizontal, axis=1) — kam common, lekin useful:

In [4]:
info1 = pd.DataFrame({'Customer ID': [12346, 12347], 'Country': ['UK', 'Germany']})
info2 = pd.DataFrame({'Total_Spend': [5000, 3000], 'Churned': [0, 1]})

horizontal_combined = pd.concat([info1, info2], axis=1)
print(horizontal_combined)

   Customer ID  Country  Total_Spend  Churned
0        12346       UK         5000        0
1        12347  Germany         3000        1


Warning: axis=1 sirf tab safe hai jab dono DataFrames ka row order aur count exactly same ho — warna galat rows aapas mein jud jayengi. Isiliye jab tak zaroori na ho, merge() (key-based) zyada safe hota hai concat(axis=1) se.

5. Different columns wale DataFrames concat karna (mismatched columns):

In [5]:
df_a = pd.DataFrame({'Customer ID': [1, 2], 'Country': ['UK', 'India']})
df_b = pd.DataFrame({'Customer ID': [3, 4], 'Country': ['USA', 'Spain'], 'Churned': [1, 0]})

mismatched = pd.concat([df_a, df_b], ignore_index=True)
print(mismatched)

   Customer ID Country  Churned
0            1      UK      NaN
1            2   India      NaN
2            3     USA      1.0
3            4   Spain      0.0


- Note: Jo column kisi ek DataFrame mein nahi tha (Churned df_a mein nahi tha), uske liye automatically NaN aa jaata hai — yeh SQL ke UNION se alag hai (SQL mein columns exactly match hone chahiye).

6. keys parameter — pata rakhna ki kaunsi row kis original DataFrame se aayi:

In [6]:
tracked = pd.concat([batch1, batch2], keys=['batch1', 'batch2'])
print(tracked)

          Customer ID         Country
batch1 0        12346  United Kingdom
       1        12347         Germany
batch2 0        12348          France
       1        12349           Spain


7. Practical exercise — apna dataset ko artificially 2 halves mein split karke phir concat se wapas jodna (verify karne ke liye ki concat sahi kaam karta hai):

In [7]:
df = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str})

half1 = df.iloc[:len(df)//2]
half2 = df.iloc[len(df)//2:]

rejoined = pd.concat([half1, half2], ignore_index=True)
print(f"Original: {df.shape}, Rejoined: {rejoined.shape}")
print(f"Shapes match: {df.shape == rejoined.shape}")

Original: (779495, 13), Rejoined: (779495, 13)
Shapes match: True


Practice questions:

1. Apna df ko Country ke hisaab se 2 groups mein manually split karo (jaise df[df['Country']=='United Kingdom'] aur baaki sab), phir concat() se wapas jodo aur verify karo ki total rows match hote hain.

In [10]:
import pandas as pd

# 1. Main DataFrame load karo
df = pd.read_csv('../data/processed/step4_datatypes_fixed.csv', dtype={'Invoice': str, 'StockCode': str})

# 2. DataFrame ko 2 groups mein split karo
df_uk = df[df['Country'] == 'United Kingdom']
df_non_uk = df[df['Country'] != 'United Kingdom']

print(f"UK rows: {df_uk.shape[0]}, Non-UK rows: {df_non_uk.shape[0]}")

# 3. Concat karke wapas jodo
df_rejoined = pd.concat([df_uk, df_non_uk], ignore_index=True)

# 4. Verify karo
print(f"Original shape: {df.shape}")
print(f"Rejoined shape: {df_rejoined.shape}")
print(f"Row counts match: {df.shape[0] == df_rejoined.shape[0]}")

UK rows: 700434, Non-UK rows: 79061
Original shape: (779495, 13)
Rejoined shape: (779495, 13)
Row counts match: True


2. keys parameter use karke pata karo ki concat ke baad kaunsi rows "UK batch" se thi aur kaunsi "non-UK batch" se.

In [11]:
# Concat with keys
df_tracked = pd.concat([df_uk, df_non_uk], keys=['UK_Batch', 'Non_UK_Batch'])

print("--- MultiIndex DataFrame with Keys ---")
print(df_tracked.head())

# Specific batch ki rows access karna `.loc` ke zariye
uk_subset = df_tracked.loc['UK_Batch']
print("\n--- Rows from UK Batch only ---")
print(uk_subset.head(3))

--- MultiIndex DataFrame with Keys ---
           Invoice StockCode                          Description  Quantity  \
UK_Batch 0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
         1  489434    79323P                   PINK CHERRY LIGHTS        12   
         2  489434    79323W                  WHITE CHERRY LIGHTS        12   
         3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
         4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

                    InvoiceDate  Price  Customer ID         Country  Year  \
UK_Batch 0  2009-12-01 07:45:00   6.95        13085  United Kingdom  2009   
         1  2009-12-01 07:45:00   6.75        13085  United Kingdom  2009   
         2  2009-12-01 07:45:00   6.75        13085  United Kingdom  2009   
         3  2009-12-01 07:45:00   2.10        13085  United Kingdom  2009   
         4  2009-12-01 07:45:00   1.25        13085  United Kingdom  2009   

            Month  Day 